# finance / portfolio / excess

Risk adjusted excess return of a portfolio at a given volatility.

| Term | In This Notebook |
|---|---|
| **Intent** | Show what is left of a portfolio's excess return once its volatility is charged for, so portfolios of different risk compare on one number. Placeholder: the excess return and risk aversion are recovered from synthetic points |
| **Project Flavour** | `pyfunc`, local to the notebook (`model.md` 3.2) |
| **Data Splits** | `training` ten volatilities with their risk adjusted excess return; `validation` five further; `backtest` ten across the range, held back for acceptance; `golden` the exact volatilities that become `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json
from os.path import join

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/portfolio/excess"
MODEL_NAME = "finance_portfolio_excess"
TARGETS = [{"kind": "local"}]
TRAINING = "portfolio_excess_training_2026-11-05_2026-11-05"
VALIDATION = "portfolio_excess_validation_2026-11-05_2026-11-05"
BACKTEST = "portfolio_excess_backtest_2026-11-05_2026-11-05"
GOLDEN = "portfolio_excess_golden"
INPUTS = ["finance_portfolio_excess_volatility_pct"]
OUTPUTS = ["finance_portfolio_excess_risk_adjusted_pct"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is synthetic, a 4 percent excess return at a risk aversion of 3, so no query runs.

In [ ]:
def adjusted(volatility, excess=4.0, gamma=3.0):
    return excess - 0.5 * gamma * (volatility / 100.0) ** 2 * 100.0


def labelled(volatilities):
    return {INPUTS[0]: volatilities, OUTPUTS[0]: [round(adjusted(volatility), 6) for volatility in volatilities]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled([5.0, 8.0, 10.0, 12.0, 15.0, 16.0, 18.0, 20.0, 22.0, 25.0]))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled([6.0, 9.0, 14.0, 19.0, 24.0]))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled([7.0, 11.0, 13.0, 17.0, 21.0, 23.0, 26.0, 28.0, 30.0, 32.0]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** regress the adjusted return on the variance in `training` only (R6): the intercept is the excess return and the slope is half the risk aversion.

In [ ]:
import numpy as np

variance = (training_pl[INPUTS[0]].to_numpy() / 100.0) ** 2 * 100.0
slope, intercept = np.polyfit(variance, training_pl[OUTPUTS[0]].to_numpy(), 1)
PARAMS = {"excess": round(float(intercept), 4), "gamma": round(float(-2.0 * slope), 4)}
print("fitted", PARAMS)
assert PARAMS == {"excess": 4.0, "gamma": 3.0}, "the fit did not recover the excess return and risk aversion"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** write `model.py` from the fitted values, build the signature, run it through the library runtime and check the `validation` and `backtest` error against the limit the notebook owns (R4, R14).

In [ ]:
from workbench.model import build_signature, evaluate_acceptance, measure, signature_input, signature_output
from workbench.replay import model_predict

MODEL_TEMPLATE = """import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = "finance_portfolio_excess_volatility_pct"
OUTPUT = "finance_portfolio_excess_risk_adjusted_pct"
EXCESS = {excess!r}
GAMMA = {gamma!r}


class Excess(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        return pd.DataFrame({{OUTPUT: [EXCESS - 0.5 * GAMMA * (volatility / 100.0) ** 2 * 100.0 for volatility in model_input[INPUT]]}})


set_model(Excess())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS))

signature = build_signature(
    MODEL_NAME,
    [
        signature_input("finance_portfolio_excess_volatility_pct", "percent", "Annualised volatility of the portfolio, percent."),
    ],
    [
        signature_output("finance_portfolio_excess_risk_adjusted_pct", "percent", "Excess return over cash less the charge for volatility, percent per year.", datatype="FP64"),
    ],
)
predict = model_predict(model_path, "pyfunc", signature)

In [ ]:
LIMIT = 0.01
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({INPUTS[0]: frame[INPUTS[0]].to_list()}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMIT, "direction": "max"}
        for split, frame in SPLITS
        for output in OUTPUTS
    ]


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package

- **Stage:** 5, package.
- **Intent:** write the golden file from the shipped runtime, copy `model.py` and the evaluation into a project, verify it and replay the golden through the library runtime (R1, R3, R14).

In [ ]:
def golden_samples():
    xs = [0.0, 5.0, 10.0, 15.0, 20.0, 30.0]
    answers = predict({INPUTS[0]: xs})
    return {INPUTS[0]: [*xs, None], **{output: [*answers[output], None] for output in OUTPUTS}, "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic risk adjusted excess returns, training split",
        [training_path, golden_path],
        "least squares on the variance",
        implementation="mlflow.pyfunc model.py, requires mlflow and pandas",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="from no volatility to a high one",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")